# 04 — Multilingual Support: 70 Languages

## Overview

Gemini Live supports **70+ languages** natively — no translation layer, no preprocessing.
The model detects language automatically or responds in whichever language you configure.

### Why this matters

| Use case | Benefit |
|---|---|
| Global customer support | One agent, 70 languages |
| Language learning apps | Native-fluency feedback |
| Regional accessibility | Users interact in their first language |
| Code-switching detection | Model follows the user mid-conversation |
| Localized voice assistants | Language-specific TTS voices |

### How language affects tone and voice

Language isn't just words — it carries **formality registers** (you vs. vous vs. usted),
**script systems** (Latin, Devanagari, Kanji, Arabic), and **cultural politeness norms**.
Gemini handles all of this automatically.

When using **audio output**, you can also configure a `language_code` hint
in `SpeechConfig` so the TTS engine is primed for the correct phoneme set.

This notebook demonstrates:
1. Sending the same question in 5 languages and collecting responses
2. Live language switching within one session
3. Automatic language detection from user input

## Setup

In [ ]:
# !pip install -q google-genai numpy

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import os
import numpy as np
import IPython.display as ipd
from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL   = "gemini-3.1-flash-live-preview"

client = genai.Client(api_key=API_KEY)
print(f"SDK ready. Model: {MODEL}")

In [ ]:
def make_pcm(duration=2.0, rate=16000, freq=440):
    """Generate sine-wave PCM bytes (int16)."""
    t = np.linspace(0, duration, int(rate * duration))
    return (np.sin(2 * np.pi * freq * t) * 0.3 * 32767).astype(np.int16).tobytes()


def play_pcm(raw_bytes, rate=24000):
    """Return an IPython Audio widget from raw PCM bytes."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


print("Audio helpers ready.")

---
## Languages Reference Table

Gemini Live supports 70+ languages. Here are 20 commonly used ones with their BCP-47 language codes (used in `SpeechConfig`) and example greetings:

| Language | Code | Greeting |
|---|---|---|
| English | `en-US` | Hello, how are you? |
| Hindi | `hi-IN` | नमस्ते, आप कैसे हैं? |
| Spanish | `es-ES` | Hola, ¿cómo estás? |
| French | `fr-FR` | Bonjour, comment allez-vous? |
| Portuguese | `pt-BR` | Olá, como vai você? |
| Arabic | `ar-XA` | مرحبا، كيف حالك؟ |
| Japanese | `ja-JP` | こんにちは、お元気ですか？ |
| Korean | `ko-KR` | 안녕하세요, 어떻게 지내세요? |
| Mandarin Chinese | `cmn-CN` | 你好，你怎么样？ |
| German | `de-DE` | Hallo, wie geht es Ihnen? |
| Russian | `ru-RU` | Здравствуйте, как вы? |
| Italian | `it-IT` | Ciao, come stai? |
| Dutch | `nl-NL` | Hallo, hoe gaat het? |
| Polish | `pl-PL` | Cześć, jak się masz? |
| Turkish | `tr-TR` | Merhaba, nasılsınız? |
| Vietnamese | `vi-VN` | Xin chào, bạn khỏe không? |
| Thai | `th-TH` | สวัสดี คุณเป็นอย่างไรบ้าง? |
| Indonesian | `id-ID` | Halo, apa kabar? |
| Bengali | `bn-IN` | হ্যালো, আপনি কেমন আছেন? |
| Swahili | `sw-KE` | Habari, hujambo? |

---
## Demo 1 — Same Question in 5 Languages

We ask "What is the capital of France?" in five languages and collect the model's text responses.
Each language gets its own session to ensure clean context.

In [ ]:
# Define the test queries — same question, 5 languages
LANGUAGE_QUERIES = [
    ("English",   "en-US", "What is the capital of France?"),
    ("Hindi",     "hi-IN", "फ्रांस की राजधानी क्या है?"),
    ("Spanish",   "es-ES", "¿Cuál es la capital de Francia?"),
    ("French",    "fr-FR", "Quelle est la capitale de la France?"),
    ("Japanese",  "ja-JP", "フランスの首都はどこですか？"),
]

print(f"Testing {len(LANGUAGE_QUERIES)} languages.")

In [ ]:
async def ask_in_language(language_name: str, query: str) -> str:
    """
    Open a single-turn session and collect the transcript response.

    Args:
        language_name: Human-readable name (for display)
        query: The question in the target language

    Returns:
        The model's transcript response
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
    )

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=query)

        response_text = ""
        async for resp in session.receive():
            if resp.server_content:
                sc = resp.server_content
                if sc.output_transcription and sc.output_transcription.text:
                    response_text += sc.output_transcription.text
                if sc.turn_complete:
                    break
            if resp.go_away:
                break

    return response_text


print("Helper function defined.")


In [ ]:
async def demo_multilingual_queries():
    """
    Ask the same question in 5 languages, display side-by-side results.
    Sessions run sequentially (could be parallelised with asyncio.gather).
    """
    results = []

    for lang_name, lang_code, query in LANGUAGE_QUERIES:
        print(f"  Querying in {lang_name}...", end=" ", flush=True)
        response = await ask_in_language(lang_name, query)
        results.append((lang_name, lang_code, query, response))
        print("done")

    # Display results
    print("\n" + "=" * 60)
    print("RESULTS — 'What is the capital of France?'")
    print("=" * 60)
    for lang_name, lang_code, query, response in results:
        print(f"\n[{lang_name} / {lang_code}]")
        print(f"  Q: {query}")
        print(f"  A: {response}")


asyncio.run(demo_multilingual_queries())

---
## Demo 2 — Language Switching Mid-Session

In this demo, a **single session** receives messages in different languages.
The system prompt instructs the model to match the user's language automatically.

This simulates a real-world scenario where a user might switch languages naturally.

In [ ]:
async def demo_language_switching():
    """
    Multi-turn session where the model follows the user's language.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        system_instruction=(
            "You are a multilingual assistant. "
            "Always respond in the same language the user uses. "
            "Do not translate — stay in the user's chosen language."
        ),
    )

    messages = [
        ("English",   "Tell me a fun fact about the ocean."),
        ("French",    "Maintenant, dis-moi quelque chose d'intéressant sur les montagnes."),
        ("Spanish",   "Cuéntame algo sobre el desierto del Sahara."),
        ("Hindi",     "मुझे भारत के बारे में कुछ बताओ।"),
        ("English",   "Back to English — summarise the themes we covered."),
    ]

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        for lang_name, msg in messages:
            print(f"\n[User in {lang_name}]: {msg}")
            await session.send_realtime_input(text=msg)

            response_text = ""
            async for resp in session.receive():
                if resp.server_content:
                    sc = resp.server_content
                    if sc.output_transcription and sc.output_transcription.text:
                        response_text += sc.output_transcription.text
                    if sc.turn_complete:
                        break
                if resp.go_away:
                    return

            print(f"[Model in {lang_name}]: {response_text}")


asyncio.run(demo_language_switching())


---
## Demo 3 — Language Detection

We send greetings in three different scripts and show the model responds in the correct language automatically — no language code specified.

In [ ]:
# Greetings in three languages — sent without any language hint
DETECTION_TESTS = [
    ("French",   "Bonjour! Comment puis-je apprendre l'intelligence artificielle?"),
    ("Spanish",  "Hola! ¿Me puedes explicar qué es el aprendizaje automático?"),
    ("Hindi",    "नमस्ते! कृत्रिम बुद्धिमत्ता क्या है?"),
]


async def demo_language_detection():
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        system_instruction="You are a helpful assistant. Respond in whatever language the user writes in.",
    )

    print("Language Detection Demo")
    print("=" * 50)

    for expected_lang, greeting in DETECTION_TESTS:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            print(f"\nInput ({expected_lang}): {greeting}")
            await session.send_realtime_input(text=greeting)

            response_text = ""
            async for resp in session.receive():
                if resp.server_content:
                    sc = resp.server_content
                    if sc.output_transcription and sc.output_transcription.text:
                        response_text += sc.output_transcription.text
                    if sc.turn_complete:
                        break
                if resp.go_away:
                    break

            print(f"Response: {response_text[:200]}..." if len(response_text) > 200 else f"Response: {response_text}")


asyncio.run(demo_language_detection())


---
## Demo 4 — Voice Configuration with Language Hint

When using **audio output**, you can set a `language_code` in `SpeechConfig`.
This primes the TTS engine for the correct phoneme inventory and prosody.

The Gemini Live API offers several pre-built voices. Here we use **Aoede** with French.

In [ ]:
# Available pre-built voices (as of mid-2025)
VOICES = [
    "Aoede",    # Default, warm female
    "Charon",   # Deep male
    "Fenrir",   # Authoritative male
    "Kore",     # Calm female
    "Puck",     # Energetic male
]

print(f"Available voices: {VOICES}")

In [ ]:
async def demo_voice_language(text: str, language_code: str, voice_name: str = "Aoede"):
    """
    Generate an audio response with a specific language/voice configuration.

    Args:
        text: The text to send to the model
        language_code: BCP-47 code, e.g. 'fr-FR', 'es-ES', 'hi-IN'
        voice_name: One of the pre-built voice names

    Returns:
        IPython Audio widget
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice_name)
            ),
            language_code=language_code,  # TTS language hint
        ),
        output_audio_transcription=types.AudioTranscriptionConfig(),  # Get text alongside audio
    )

    audio_chunks = []
    transcript = ""

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=text)

        async for resp in session.receive():
            # Collect audio data
            if resp.data:
                audio_chunks.append(resp.data)

            if resp.server_content:
                sc = resp.server_content
                # Collect transcription
                if sc.output_transcription and sc.output_transcription.text:
                    transcript += sc.output_transcription.text
                if sc.turn_complete:
                    break

            if resp.go_away:
                break

    if not audio_chunks:
        print("No audio received.")
        return None

    raw_audio = b"".join(audio_chunks)
    print(f"Language: {language_code} | Voice: {voice_name}")
    print(f"Transcript: {transcript}")
    print(f"Audio: {len(raw_audio):,} bytes ({len(raw_audio)//2:,} samples)")

    return play_pcm(raw_audio, rate=24000)


print("Voice demo function ready.")

In [ ]:
# Demo: French response with Aoede voice
audio = asyncio.run(demo_voice_language(
    text="Bonjour! Peux-tu me parler de la Tour Eiffel en quelques phrases?",
    language_code="fr-FR",
    voice_name="Aoede",
))
audio  # Display the audio player

In [ ]:
# Demo: Spanish response with Puck voice
audio_es = asyncio.run(demo_voice_language(
    text="Hola! Cuéntame algo interesante sobre España.",
    language_code="es-ES",
    voice_name="Puck",
))
audio_es

In [ ]:
# Demo: Hindi response with Kore voice
audio_hi = asyncio.run(demo_voice_language(
    text="नमस्ते! भारत की संस्कृति के बारे में कुछ बताइए।",
    language_code="hi-IN",
    voice_name="Kore",
))
audio_hi

---
## Voice + Language Code Quick Reference

```python
# Full pattern for audio output with language hint
config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Aoede")
        ),
        language_code="fr-FR",  # BCP-47 code primes TTS for correct phonemes
    ),
    output_audio_transcription=types.AudioTranscriptionConfig(),
)
```

### Common language codes:

| Language | Code | Notes |
|---|---|---|
| English (US) | `en-US` | Default |
| English (UK) | `en-GB` | British accent TTS |
| French | `fr-FR` | Metropolitan French |
| Spanish (Spain) | `es-ES` | Castilian |
| Spanish (LATAM) | `es-419` | Latin American |
| Hindi | `hi-IN` | Devanagari script |
| Japanese | `ja-JP` | |
| Korean | `ko-KR` | |
| Mandarin | `cmn-CN` | Simplified Chinese |
| Arabic | `ar-XA` | MSA (Modern Standard Arabic) |
| Portuguese (BR) | `pt-BR` | Brazilian |
| German | `de-DE` | |

### Notes:
- For **text-only** responses, no `language_code` is needed — auto-detection is excellent
- For **audio output**, providing `language_code` improves TTS prosody and accent accuracy
- The model's comprehension works without a code; the code only affects TTS output
- `output_audio_transcription` gives you text alongside audio — useful for logging/testing

---
## Key Takeaways

1. **No translation pipeline needed** — Gemini Live understands and speaks 70+ languages natively

2. **Auto-detection works well** — for text input, the model automatically detects language; no config needed

3. **System prompt controls behaviour** — "Respond in whatever language the user uses" enables seamless code-switching

4. **`language_code` in `SpeechConfig` improves TTS** — when generating audio, set this to the expected output language

5. **`output_audio_transcription`** — always enable this in demos so you can verify audio content via text

6. **Multi-turn sessions maintain context** — language history is preserved across turns in a single session

### Use cases to explore:
- Live translation: user speaks in Hindi, model responds in English
- Language tutor: user writes in imperfect French, model corrects and responds in French
- Multilingual IVR: detect user language from first utterance, route accordingly
- Code-switching detection for Singapore English / Singlish / Mandarin mixing